Importação de bibliotecas

## Etapa 1 - Parametros do experimento da pandemia

Execute a celula abaixo para conferir as datas e a pasta de dados, sem baixar nada. As datas ficam em `parametros_experimento.json`, na pasta do projeto.

Sua parte utiliza 2017-2019 para construir a MST e selecionar os ativos. O periodo 2020-2022 e uma referencia para entrega ao integrante responsavel pela avaliacao financeira. O planejamento esta em `PLANO_ALGORITMOS.md`.

Esta etapa ainda nao recorta Pearson nem seleciona ativos elegiveis.


In [24]:
from pathlib import Path
import json

parametros = json.loads(Path('parametros_experimento.json').read_text(encoding='utf-8'))
inicio_formacao = parametros['inicio_formacao']
fim_formacao = parametros['fim_formacao']
inicio_avaliacao = parametros['inicio_avaliacao']
fim_avaliacao = parametros['fim_avaliacao']

print('Formacao da MST:', inicio_formacao, 'ate', fim_formacao)
print('Avaliacao financeira (outro integrante):', inicio_avaliacao, 'ate', fim_avaliacao)
pasta_coleta = Path(parametros['pasta_coleta'])
print('Pasta da coleta:', pasta_coleta.resolve())
if (pasta_coleta / 'acoes.csv').exists() and (pasta_coleta / 'metadados.json').exists():
    print('Coleta local encontrada; a celula de dados verificara sua integridade.')
else:
    print('Coleta local ainda nao esta completa. Esta celula nao realiza downloads.')


Formacao da MST: 2017-01-01 ate 2019-12-31
Avaliacao financeira (outro integrante): 2020-01-01 ate 2022-12-31
Pasta da coleta: C:\Users\cida3\OneDrive\Documentos\TrabalhoDiscreta\dados\pandemia
Coleta local ainda nao esta completa. Esta celula nao realiza downloads.


### Continuacao: importacoes e dados
A celula acima funciona sozinha. Executar as celulas seguintes pode instalar bibliotecas e consultar servicos externos.

In [25]:
import numpy as np
import pandas as pd

%run ./Tabelas_Acoes.ipynb

Formacao da MST: 2017-01-01 ate 2019-12-31
Avaliacao financeira (outro integrante): 2020-01-01 ate 2022-12-31
Pasta da coleta: C:\Users\cida3\OneDrive\Documentos\TrabalhoDiscreta\dados\pandemia
Coleta local ainda nao esta completa. Esta celula nao realiza downloads.


$CASN3.SA: possibly delisted; no price data found  (1d 2008-01-01 -> 2026-01-01)
$CASN4.SA: possibly delisted; no price data found  (1d 2008-01-01 -> 2026-01-01)
$AZUL3.SA: Data doesn't exist for startDate = 1199152800, endDate = 1767236400
$AUAU3.SA: Data doesn't exist for startDate = 1199152800, endDate = 1767236400

4 Failed downloads:
['CASN3.SA', 'CASN4.SA']: possibly delisted; no price data found  (1d 2008-01-01 -> 2026-01-01)
['AZUL3.SA', 'AUAU3.SA']: Data doesn't exist for startDate = 1199152800, endDate = 1767236400
$CYRE4.SA: Data doesn't exist for startDate = 1199152800, endDate = 1767236400
$HETA3.SA: Data doesn't exist for startDate = 1199152800, endDate = 1767236400
$EGGY3.SA: Data doesn't exist for startDate = 1199152800, endDate = 1767236400

3 Failed downloads:
['CYRE4.SA', 'HETA3.SA', 'EGGY3.SA']: Data doesn't exist for startDate = 1199152800, endDate = 1767236400
$ODER4.SA: Data doesn't exist for startDate = 1199152800, endDate = 1767236400
$PASS3.SA: Data doesn't ex

Nova coleta salva: dados\pandemia\acoes.csv
Data da coleta (UTC): 2026-11-16T22:11:41.016388+00:00


c:\Users\cida3\anaconda3\envs\AED082026\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)
c:\Users\cida3\anaconda3\envs\AED082026\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)


## Etapa 2 - Elegibilidade e retornos da formacao (2017-2019)

Execute as duas celulas abaixo depois de carregar `df` e `acoes`, com NumPy e Pandas importados. Se os dados ja estao no kernel, nao e necessario repetir o `%run`.

A cobertura minima esta em `parametros_experimento.json`. As datas da base sao uma referencia provisoria de sessoes, nao um calendario oficial da B3. A funcao permite receber um calendario pelo argumento `sessoes`.

`Elegivel` significa que a acao atende aos criterios observaveis nos dados; nao certifica a data do IPO. Falta de evidencia no inicio, ausencia de preco na ultima sessao ou precos invalidos geram pendencia, salvo se ja houver motivo numerico de exclusao. Mudancas de ticker e o universo historico precisam de verificacao com o integrante responsavel pelos dados.

A base `df` e preservada. Sao criados `tabela_elegibilidade`, `retornos_formacao`, `acoes_elegiveis` e `retornos_formacao_elegiveis`. A tabela inclui tambem tickers solicitados sem fechamento disponivel. Nao sao usados precos posteriores a 2019. A tabela e os retornos ficam em memoria; a exportacao pertence a etapa 7.

**Confira a tabela antes de prosseguir. As etapas seguintes usam apenas os retornos elegiveis desta formacao.**


In [39]:
def preparar_formacao(precos, inicio, fim, cobertura_minima=0.9,
                      tickers_solicitados=None, sessoes=None):
    if not 0 < cobertura_minima <= 1:
        raise ValueError('A cobertura minima deve estar entre 0 e 1.')
    if not isinstance(precos.index, pd.DatetimeIndex) or not precos.index.is_unique:
        raise ValueError('Os precos precisam de datas unicas em um DatetimeIndex.')
    if not precos.columns.is_unique:
        raise ValueError('Os tickers nao podem estar duplicados.')
    inicio, fim = pd.Timestamp(inicio), pd.Timestamp(fim)
    if inicio > fim:
        raise ValueError('O inicio da formacao deve ser anterior ao fim.')

    # Descarta o futuro antes de qualquer decisao de elegibilidade.
    precos = precos.sort_index().loc[:fim].copy()
    tickers = set(precos.columns)
    if tickers_solicitados is not None:
        tickers.update(tickers_solicitados)
    precos = precos.reindex(columns=sorted(tickers))

    referencia = 'datas da base (provisorio)' if sessoes is None else 'calendario fornecido'
    if sessoes is None:
        sessoes = precos.index
    sessoes = pd.DatetimeIndex(sessoes).unique().sort_values()
    sessoes = sessoes[(sessoes >= inicio) & (sessoes <= fim)]
    if len(sessoes) == 0:
        raise ValueError('Nao ha datas de referencia no periodo de formacao.')

    # Insere sessoes ausentes antes da diferenca, sem preencher precos.
    precos = precos.reindex(precos.index.union(sessoes).sort_values())
    invalidos = precos.notna() & (~np.isfinite(precos) | (precos <= 0))
    precos_validos = precos.where(np.isfinite(precos) & (precos > 0))
    precos_janela = precos_validos.reindex(sessoes)
    # O primeiro retorno pode usar o fechamento anterior a 2017.
    retornos = np.log(precos_validos).diff().reindex(sessoes)

    tabela = pd.DataFrame({
        'Primeiro_preco_na_formacao': precos_janela.apply(lambda serie: serie.first_valid_index()),
        'Ultimo_preco_na_formacao': precos_janela.apply(lambda serie: serie.last_valid_index()),
        'Historico_no_inicio': precos_validos.loc[:sessoes[0]].notna().any(),
        'Preco_na_ultima_sessao': precos_janela.iloc[-1].notna(),
        'Retornos_validos': retornos.count(),
        'Sessoes_referencia': len(sessoes),
        'Cobertura': retornos.count() / len(sessoes),
        'Variancia': retornos.var(),
        'Precos_invalidos': invalidos.reindex(sessoes).sum()
    })
    tabela.index.name = 'Ticker'
    tabela['Situacao'] = 'Elegivel'
    tabela['Motivo'] = 'Atende aos criterios observaveis na base; listagem oficial nao verificada.'
    for ticker, linha in tabela.iterrows():
        exclusoes, pendencias = [], []
        if linha['Retornos_validos'] < 2:
            exclusoes.append('Menos de dois retornos validos na formacao')
        elif not np.isfinite(linha['Variancia']) or linha['Variancia'] <= 0:
            exclusoes.append('Variancia nula ou indefinida')
        if linha['Cobertura'] < cobertura_minima:
            exclusoes.append('Cobertura abaixo do minimo')
        if not linha['Historico_no_inicio']:
            pendencias.append('Sem evidencia de negociacao no inicio; investigar historico ou ticker')
        if not linha['Preco_na_ultima_sessao']:
            pendencias.append('Sem preco na ultima sessao de referencia; investigar')
        if linha['Precos_invalidos'] > 0:
            pendencias.append('Precos nao positivos ou infinitos na formacao; investigar')
        if exclusoes or pendencias:
            tabela.loc[ticker, 'Situacao'] = 'Excluida' if exclusoes else 'Pendente'
            tabela.loc[ticker, 'Motivo'] = '; '.join(exclusoes + pendencias)
    tabela.attrs['referencia_calendario'] = referencia
    return tabela, retornos


In [40]:
# Pode ser executada com df ja carregado, sem repetir o %run/download.
from pathlib import Path
import json

parametros = json.loads(Path('parametros_experimento.json').read_text(encoding='utf-8'))
inicio_formacao = parametros['inicio_formacao']
fim_formacao = parametros['fim_formacao']
tabela_elegibilidade, retornos_formacao = preparar_formacao(
    df['Close'], inicio_formacao, fim_formacao,
    cobertura_minima=parametros['cobertura_minima_formacao'],
    tickers_solicitados=acoes
)
acoes_elegiveis = tabela_elegibilidade.index[
    tabela_elegibilidade['Situacao'] == 'Elegivel'
].tolist()
retornos_formacao_elegiveis = retornos_formacao[acoes_elegiveis].copy()

print('Formacao:', inicio_formacao, 'ate', fim_formacao)
print('Referencia das sessoes:', tabela_elegibilidade.attrs['referencia_calendario'])
print(tabela_elegibilidade['Situacao'].value_counts())
display(tabela_elegibilidade)
print('Entrada da etapa 3:', retornos_formacao_elegiveis.shape, '(datas, ativos elegiveis)')


Formacao: 2017-01-01 ate 2019-12-31
Referencia das sessoes: datas da base (provisorio)
Situacao
Elegivel    275
Excluida    102
Pendente      2
Name: count, dtype: int64


,Primeiro_preco_na_formacao,Ultimo_preco_na_formacao,Historico_no_inicio,Preco_na_ultima_sessao,Retornos_validos,Sessoes_referencia,Cobertura,Variancia,Precos_invalidos,Situacao,Motivo
Ticker,,,,,,,,,,,
AALR3.SA,2017-01-02,2019-12-30,True,True,748,748,1.00000,0.000274,0,Elegivel,Atende aos criterios observaveis na base; list...
ABCB4.SA,2017-01-02,2019-12-30,True,True,748,748,1.00000,0.000294,0,Elegivel,Atende aos criterios observaveis na base; list...
ABEV3.SA,2017-01-02,2019-12-30,True,True,748,748,1.00000,0.000194,0,Elegivel,Atende aos criterios observaveis na base; list...
AERI3.SA,NaT,NaT,False,False,0,748,0.00000,NaN,0,Excluida,Menos de dois retornos validos na formacao; Co...
AFLT3.SA,2017-01-02,2019-12-30,True,True,748,748,1.00000,0.004316,0,Elegivel,Atende aos criterios observaveis na base; list...
...,...,...,...,...,...,...,...,...,...,...,...
WHRL4.SA,2017-01-02,2019-12-30,True,True,748,748,1.00000,0.000705,0,Elegivel,Atende aos criterios observaveis na base; list...
WIZC3.SA,2017-01-02,2019-12-30,True,True,748,748,1.00000,0.000716,0,Elegivel,Atende aos criterios observaveis na base; list...
WLMM3.SA,2017-07-07,2019-12-30,False,True,619,748,0.82754,0.001762,0,Excluida,Cobertura abaixo do minimo; Sem evidencia de n...


Entrada da etapa 3: (748, 275) (datas, ativos elegiveis)


## Etapa 3 - Pearson e distancias de Mantegna da formacao

As matrizes abaixo usam somente `retornos_formacao_elegiveis` de 2017-2019. O minimo de observacoes conjuntas por par e definido no arquivo de parametros. As funcoes de Kruskal e metricas existentes serao reaplicadas nas etapas 4 e 5; Prim permanece sem chamada ativa.


In [41]:
# Correlacoes dos retornos elegiveis da formacao, sem dados da avaliacao.
min_observacoes_conjuntas = int(np.ceil(
    len(retornos_formacao_elegiveis)
    * parametros['proporcao_minima_observacoes_conjuntas']
))
matriz_correlacoes_formacao = retornos_formacao_elegiveis.corr(
    method='pearson', min_periods=min_observacoes_conjuntas
)
matriz_correlacoes_formacao


Ticker,AALR3.SA,ABCB4.SA,ABEV3.SA,AFLT3.SA,AGRO3.SA,AHEB3.SA,AHEB5.SA,AHEB6.SA,ALOS3.SA,ALPA3.SA,...,VIVT3.SA,VLID3.SA,VSTE3.SA,VULC3.SA,WEGE3.SA,WHRL3.SA,WHRL4.SA,WIZC3.SA,WLMM4.SA,YDUQ3.SA
Ticker,,,,,,,,,,,,,,,,,,,,,
AALR3.SA,1.000000,0.202691,0.110106,-0.013838,0.083588,-0.020221,-0.024858,-0.067121,0.121380,0.101798,...,0.013557,0.085702,0.045009,0.111357,0.046897,-0.028191,-0.001200,0.109219,0.090936,0.126002
ABCB4.SA,0.202691,1.000000,0.189649,0.033669,0.220783,0.015122,0.048736,0.048607,0.261322,0.068491,...,0.168491,0.264149,0.185372,0.219496,0.297666,0.053546,0.014184,0.141744,0.064299,0.366786
ABEV3.SA,0.110106,0.189649,1.000000,-0.014037,0.114370,-0.028529,-0.015325,-0.025505,0.180407,0.077327,...,0.203723,0.114467,0.076083,0.141875,0.298130,0.047015,-0.022566,0.095804,0.044337,0.256555
AFLT3.SA,-0.013838,0.033669,-0.014037,1.000000,-0.024666,0.001887,0.020043,-0.000938,-0.004160,0.016907,...,0.010453,0.052157,0.027905,0.020164,0.022700,0.037199,-0.018321,0.021938,-0.002654,0.031697
AGRO3.SA,0.083588,0.220783,0.114370,-0.024666,1.000000,0.007773,0.028699,0.120076,0.143612,0.053625,...,0.171984,0.176458,0.112741,0.147901,0.114666,0.023258,-0.024831,0.136881,-0.001942,0.221729
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WHRL3.SA,-0.028191,0.053546,0.047015,0.037199,0.023258,0.001194,0.026041,0.005335,0.121402,-0.037461,...,0.111376,0.155517,0.001633,0.087572,0.127025,1.000000,0.464049,0.018313,0.031458,0.061316
WHRL4.SA,-0.001200,0.014184,-0.022566,-0.018321,-0.024831,0.034998,-0.008479,-0.014146,0.052264,0.037552,...,0.079260,0.020315,0.036685,0.079153,0.040334,0.464049,1.000000,-0.008069,0.071907,0.024827
WIZC3.SA,0.109219,0.141744,0.095804,0.021938,0.136881,-0.006206,0.039229,-0.003504,0.116522,-0.001103,...,0.094683,0.067608,0.127214,0.138995,0.099614,0.018313,-0.008069,1.000000,0.039860,0.212282


In [51]:
def matriz_mantegna(matriz_pearson):
     return np.sqrt(2*(1-matriz_pearson))

matriz_mantegna_formacao = matriz_mantegna(matriz_correlacoes_formacao)
matriz_mantegna_formacao


Ticker,AALR3.SA,ABCB4.SA,ABEV3.SA,AFLT3.SA,AGRO3.SA,AHEB3.SA,AHEB5.SA,AHEB6.SA,ALOS3.SA,ALPA3.SA,...,VIVT3.SA,VLID3.SA,VSTE3.SA,VULC3.SA,WEGE3.SA,WHRL3.SA,WHRL4.SA,WIZC3.SA,WLMM4.SA,YDUQ3.SA
Ticker,,,,,,,,,,,,,,,,,,,,,
AALR3.SA,0.000000,1.262782,1.334087,1.423965,1.353818,1.428440,1.431683,1.460905,1.325610,1.340300,...,1.404595,1.352256,1.382021,1.333149,1.380654,1.434009,1.415062,1.334751,1.348380,1.322118
ABCB4.SA,1.262782,0.000000,1.273068,1.390202,1.248372,1.403480,1.379322,1.379415,1.215465,1.364924,...,1.289580,1.213137,1.276423,1.249403,1.185187,1.375830,1.404148,1.310157,1.367992,1.125356
ABEV3.SA,1.334087,1.273068,0.000000,1.424105,1.330887,1.434245,1.425009,1.432135,1.280307,1.358435,...,1.261964,1.330814,1.359351,1.310057,1.184796,1.380569,1.430081,1.344764,1.382507,1.219381
AFLT3.SA,1.423965,1.390202,1.424105,0.000000,1.431549,1.412879,1.399970,1.414877,1.417152,1.402208,...,1.406803,1.376839,1.394342,1.399883,1.398070,1.387660,1.427109,1.398615,1.416089,1.391620
AGRO3.SA,1.353818,1.248372,1.330887,1.431549,0.000000,1.408706,1.393773,1.326593,1.308731,1.375773,...,1.286869,1.283388,1.332110,1.305450,1.330665,1.397671,1.431664,1.313864,1.415586,1.247615
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WHRL3.SA,1.434009,1.375830,1.380569,1.387660,1.397671,1.413369,1.395679,1.410436,1.325593,1.440459,...,1.333134,1.299602,1.413058,1.350872,1.321344,0.000000,1.035327,1.401205,1.391791,1.370171
WHRL4.SA,1.415062,1.404148,1.430081,1.427109,1.431664,1.389246,1.420196,1.424181,1.376761,1.387406,...,1.357011,1.399775,1.388031,1.357090,1.385400,1.035327,0.000000,1.419907,1.362419,1.396548
WIZC3.SA,1.334751,1.310157,1.344764,1.398615,1.313864,1.418595,1.386197,1.416689,1.329269,1.414993,...,1.345598,1.365571,1.321201,1.312254,1.341928,1.401205,1.419907,0.000000,1.385742,1.255164


In [52]:
# Ramo exploratorio de IPCA: nao utilizado na MST da formacao.
dataframe_IPCA = inflacao_mensal('01/01/2008', '31/12/2025', variacao_logaritma)
matriz_correlacao_IPCA = dataframe_IPCA.corr(method='pearson')
matriz_mantegna_IPCA = matriz_mantegna(matriz_correlacao_IPCA)
matriz_mantegna_IPCA


Ticker,AALR3.SA,ABCB4.SA,ABEV3.SA,AERI3.SA,AFLT3.SA,AGRO3.SA,AGXY3.SA,AHEB3.SA,AHEB5.SA,AHEB6.SA,...,VVEO3.SA,WDCN3.SA,WEGE3.SA,WEST3.SA,WHRL3.SA,WHRL4.SA,WIZC3.SA,WLMM3.SA,WLMM4.SA,YDUQ3.SA
Ticker,,,,,,,,,,,,,,,,,,,,,
AALR3.SA,0.000000,1.704417,0.924978,1.294628,1.454682,1.669608,0.677664,1.196351,1.172692,1.120467,...,0.745838,1.140178,1.757616,1.379627,1.524702,1.541576,1.100217,1.661687,1.722044,1.166335
ABCB4.SA,1.704417,0.000000,1.279995,1.865418,1.411735,0.704319,1.926850,1.362498,1.339858,1.225228,...,1.879404,1.843638,0.692761,1.797131,0.876629,0.889326,1.605027,1.257971,1.012812,1.230195
ABEV3.SA,0.924978,1.279995,0.000000,0.611612,1.851239,1.526964,0.809750,1.224491,1.087871,0.972039,...,0.738861,0.723292,1.434524,0.704791,1.244488,1.271774,0.950919,1.800141,1.489921,0.897991
AERI3.SA,1.294628,1.865418,0.611612,0.000000,0.746054,1.463357,0.712185,1.365848,1.596775,1.132868,...,0.785605,0.223225,1.424814,0.379281,0.345908,0.400217,0.870020,1.076707,0.997271,0.485283
AFLT3.SA,1.454682,1.411735,1.851239,0.746054,0.000000,1.209491,0.670768,1.459405,1.630692,1.583092,...,0.660648,0.988693,1.313996,0.842420,1.329453,1.306235,1.508114,1.129972,1.362239,1.478826
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WHRL4.SA,1.541576,0.889326,1.271774,0.400217,1.306235,0.788983,1.011034,1.440111,1.507873,1.224668,...,1.048162,0.443401,0.708333,0.571198,0.174015,0.000000,1.384091,1.098920,0.850889,0.903323
WIZC3.SA,1.100217,1.605027,0.950919,0.870020,1.508114,1.702367,1.126504,1.283783,1.199908,1.015482,...,1.104330,0.697013,1.690329,0.809730,1.398500,1.384091,0.000000,1.474599,1.639765,1.058242
WLMM3.SA,1.661687,1.257971,1.800141,1.076707,1.129972,0.596167,1.161864,1.808083,1.840540,1.772861,...,1.261293,0.734490,0.656380,1.107470,1.155712,1.098920,1.474599,0.000000,0.377054,1.709209


In [53]:
def validar_matriz(df):
    # NaN representa uma distancia desconhecida, nunca uma aresta de peso zero.
    if not isinstance(df, pd.DataFrame):
        raise ValueError('A matriz deve ser um DataFrame.')
    if df.empty or df.shape[0] != df.shape[1]:
        raise ValueError('A matriz deve ser quadrada e nao vazia.')
    if not df.index.equals(df.columns) or not df.index.is_unique:
        raise ValueError('Linhas e colunas devem ter os mesmos ativos, sem repeticao e na mesma ordem.')
    valores = df.to_numpy()
    if valores.dtype.kind not in 'fiu':
        raise ValueError('As distancias devem ser numeros reais.')
    if np.isinf(valores).any():
        raise ValueError('A matriz contem distancias infinitas.')
    if not np.allclose(valores, valores.T, rtol=0, atol=1e-10, equal_nan=True):
        raise ValueError('A matriz deve ser simetrica, inclusive nas posicoes ausentes.')
    diagonal = np.diag(valores)
    if not np.all(np.isnan(diagonal) | np.isclose(diagonal, 0, rtol=0, atol=1e-10)):
        raise ValueError('As distancias conhecidas na diagonal devem ser zero.')
    conhecidos = valores[np.isfinite(valores)]
    if ((conhecidos < 0) | (conhecidos > 2)).any():
        raise ValueError('As distancias de Mantegna devem estar entre 0 e 2.')


def criar_arestas(df):
    validar_matriz(df)
    n=len(df.columns)
    linha = 0
    arestas = []
    for j in range(1,n):
        for i in range(linha+1):
            peso = df.iloc[i,j]
            if np.isfinite(peso):
                arestas.append((peso,i,j ))
        linha+=1
    
    return sorted(arestas)
arestas = criar_arestas(matriz_mantegna_formacao)
print(arestas)


[(np.float64(0.32165015788673273), 191, 192), (np.float64(0.3669197609457694), 24, 25), (np.float64(0.4125141158444069), 146, 148), (np.float64(0.45618223718660983), 147, 148), (np.float64(0.4856574221456088), 48, 263), (np.float64(0.4936509747502307), 72, 73), (np.float64(0.4980522766188801), 128, 130), (np.float64(0.5183427543940982), 25, 148), (np.float64(0.5329624238899879), 47, 48), (np.float64(0.5569566642698728), 129, 130), (np.float64(0.5774446651952857), 146, 147), (np.float64(0.5836861060947132), 24, 148), (np.float64(0.5895453796237475), 25, 146), (np.float64(0.6023779215732009), 240, 241), (np.float64(0.6105579438057402), 25, 147), (np.float64(0.6254850650881919), 24, 146), (np.float64(0.6419900665369446), 24, 147), (np.float64(0.6461384198127903), 127, 128), (np.float64(0.6634751785748478), 23, 25), (np.float64(0.6744011606385426), 145, 146), (np.float64(0.6778135255373976), 47, 263), (np.float64(0.6904595002264409), 79, 80), (np.float64(0.7003817242574282), 128, 129), (np

In [54]:
def find(lista, vertice):
    if lista[vertice] == vertice:
        return vertice
    while lista[vertice] != vertice:
        vertice = lista[vertice]
    return vertice

def union(lista, vertice1, vertice2):
    pai1 = find(lista, vertice1)
    pai2 = find(lista, vertice2)
    if pai1 != pai2:
         lista[pai2] = pai1

def kruskal(df):
    arestas = criar_arestas(df)
    n = len(df.columns)
    pais = list(range(n))
    mst = []
    for peso, vertice1, vertice2 in arestas:
        if find(pais, vertice1) != find(pais, vertice2):
            union(pais, vertice1, vertice2)
            mst.append((peso, vertice1, vertice2))
        if len(mst) == n-1:
            break
    if len(mst) != n-1:
        raise ValueError('Grafo desconexo: as distancias conhecidas nao conectam todos os ativos.')
    return mst


In [55]:
def prim(df):
    validar_matriz(df)
    n = len(df.columns)
    vertice_inicial = 0
    visitados = set()
    visitados.add(vertice_inicial)
    mst = []
    while len(visitados) < n:
        menor_peso = (np.inf, None, None)
        for u in visitados:
            for v in range(n):
                if v not in visitados:
                    peso = df.iloc[u,v]
                    if np.isfinite(peso) and peso < menor_peso[0]:
                        menor_peso = (peso,u,v)
        if menor_peso[2] is None:
            raise ValueError('Grafo desconexo: as distancias conhecidas nao conectam todos os ativos.')
        visitados.add(menor_peso[2])
        mst.append(menor_peso)
    
    return mst


In [48]:
# Etapa 4 - MST da formacao; a chamada de Prim permanece comentada.
#mst_prim = prim(matriz_mantegna_formacao)
mst_kruskal = kruskal(matriz_mantegna_formacao)

#print("Arestas Prim:", len(mst_prim))
print("Arestas Kruskal:", len(mst_kruskal))

#peso_prim = sum(peso for peso, u, v in mst_prim)
peso_kruskal = sum(peso for peso, u, v in mst_kruskal)

#print("Peso total Prim:", peso_prim)
print("Peso total Kruskal:", peso_kruskal)

Arestas Kruskal: 274
Peso total Kruskal: 310.4498804204607


In [56]:
def criar_dicionario_adjacencia(mst):
    adjacencia = {}
    for peso, v1, v2 in mst:
        adjacencia.setdefault(v1,[]).append((peso,v2))
        adjacencia.setdefault(v2,[]).append((peso,v1))
    return adjacencia

def calcular_graus(adjacencia):
    graus = {}
    for vertice in adjacencia:
        graus[vertice] = len(adjacencia[vertice])
    return graus


def calcular_excentricidade(adjacencia):
    excentricidades = {}

    for origem in adjacencia:
        distancias = {origem: 0}
        pilha = [origem]

        while pilha:
            vertice = pilha.pop()

            for peso, vizinho in adjacencia[vertice]:
                if vizinho not in distancias:
                    distancias[vizinho] = distancias[vertice] + peso
                    pilha.append(vizinho)

        if len(distancias) != len(adjacencia):
            raise ValueError("A árvore precisa ser conexa.")

        excentricidades[origem] = max(distancias.values())

    return excentricidades


def distancia_entre_vertices(adjacencia, origem, destino):
    if origem not in adjacencia or destino not in adjacencia:
        raise ValueError('Os dois vertices devem pertencer a arvore.')
    distancias = {origem: 0}
    pilha = [origem]
    while pilha:
        vertice = pilha.pop()
        if vertice == destino:
            return distancias[vertice]
        for peso, vizinho in adjacencia[vertice]:
            if vizinho not in distancias:
                distancias[vizinho] = distancias[vertice] + peso
                pilha.append(vizinho)
    raise ValueError('Nao existe caminho entre os vertices informados.')


def calcular_diametro(adjacencia):
    if not adjacencia:
        raise ValueError('A arvore nao pode ser vazia.')
    excentricidades = calcular_excentricidade(adjacencia)
    return max(excentricidades.values())


def calcular_centro(adjacencia):
    if not adjacencia:
        raise ValueError('A arvore nao pode ser vazia.')
    excentricidades = calcular_excentricidade(adjacencia)
    menor_excentricidade = min(excentricidades.values())
    centro = []
    for vertice, excentricidade in excentricidades.items():
        if abs(excentricidade - menor_excentricidade) <= 1e-10:
            centro.append(vertice)
    return centro

In [57]:
# Etapa 5 - Metricas estruturais da MST da formacao.
adjacencia = criar_dicionario_adjacencia(mst_kruskal)
graus = calcular_graus(adjacencia)
excentricidades = calcular_excentricidade(adjacencia)
diametro = calcular_diametro(adjacencia)
centro = calcular_centro(adjacencia)
vertices = sorted(adjacencia)

metricas = pd.DataFrame({
    'Vertice': vertices,
    'Ticker': [matriz_mantegna_formacao.columns[v] for v in vertices],
    'Grau': [graus[v] for v in vertices],
    'Excentricidade': [excentricidades[v] for v in vertices],
    'Folha': [graus[v] == 1 for v in vertices],
    'Centro': [v in centro for v in vertices]
})

display(metricas.sort_values(['Excentricidade', 'Ticker']))
print('Diametro:', diametro)
print('Centro:', [matriz_mantegna_formacao.columns[v] for v in centro])


,Vertice,Ticker,Grau,Excentricidade,Folha,Centro
90,90,CYRE3.SA,12,13.952992,False,True
177,177,MULT3.SA,10,13.960033,False,False
117,117,EZTC3.SA,2,14.730798,False,False
116,116,EVEN3.SA,3,14.835680,False,False
159,159,LREN3.SA,6,14.840390,False,False
...,...,...,...,...,...,...
62,62,CEEB3.SA,1,25.681501,True,False
137,137,HBTS5.SA,1,25.747220,True,False
63,63,CEED3.SA,2,25.752769,False,False
175,175,MTSA3.SA,1,27.025547,True,False


Diametro: 27.025547119992865
Centro: ['CYRE3.SA']


In [ ]:
# Etapa 6 - Carteira central: oito menores excentricidades.
# Empate resolvido pelo ticker em ordem alfabetica.
carteira_central = metricas.sort_values(
    ['Excentricidade', 'Ticker'], ascending=[True, True]
).head(8).copy()

display(carteira_central)


In [ ]:
# Etapa 6 - Carteira periferica: oito maiores excentricidades.
# Empate resolvido pelo ticker em ordem alfabetica.
carteira_periferica = metricas.sort_values(
    ['Excentricidade', 'Ticker'], ascending=[False, True]
).head(8).copy()

display(carteira_periferica)
